---
## 🗒️ Notebook Overview

Welcome! This notebook walks through **LangChain Tool Calling** — a pattern where an AI model decides *which tool to call* and *what arguments to pass*, just like calling a method in Java.

> **Java parallel:** Think of tools as `@Service` beans, the LLM as an `Orchestrator`, and the agent as a `DispatcherServlet` that routes requests to the right service method.

### What you'll build step-by-step:
1. ✅ Load imports & setup
2. ✅ Connect to an LLM (the AI brain)
3. 🔧 Define custom tools (like writing Java methods)
4. 🤖 Create an Agent (the orchestrator / dispatcher)
5. 🚀 Run the agent on real tasks

---


In [1]:
# ==========================================
# Imports
# ==========================================

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="langchain_community")
warnings.filterwarnings("ignore", category=UserWarning, module="langchain_community")

from langchain_core.tools import tool
from pprint import pprint

print("✅ All imports loaded successfully")

✅ All imports loaded successfully


---
### 📦 Cell 1 — Imports & Warnings

This cell imports the core libraries and suppresses noisy deprecation warnings.

| Python | Java equivalent |
|--------|----------------|
| `import warnings` | Suppressing `@SuppressWarnings("deprecation")` |
| `from langchain_core.tools import tool` | `import com.langchain.Tool;` |
| `from pprint import pprint` | `System.out.println()` / pretty-printer |

> **Key concept:** `langchain_core` is the foundation layer — like importing from `java.util` or `spring-core`.

---


In [2]:
# ==========================================
# Setup Project Root Path
# ==========================================

import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project Root: {project_root}")

Project Root: d:\projects\python\stepIntoAI\LangChain_05-tool-calling


---
### 📁 Cell 2 — Project Root Path Setup

Python doesn't have a built-in classpath like Java. This cell manually adds the project root to `sys.path` so that our `src/` modules can be imported.

> **Java parallel:** This is similar to adding a directory to your Maven/Gradle `sourceSets` or IntelliJ's "Mark Directory as Sources Root".

```python
# Python: dynamically add to module search path
sys.path.append(str(project_root))

# Java equivalent idea: adding to classpath at runtime
# URLClassLoader loader = new URLClassLoader(new URL[]{new File("src/").toURI().toURL()});
```

---


In [3]:
# ==========================================
# Load Configuration
# ==========================================

from src.utils.config_loader import load_config
config = load_config()

print("Configuration Loaded Successfully\n")
print(config)

Configuration Loaded Successfully

{'llm': {'provider': 'gemini', 'gemini': {'model_name': 'gemini-3.6-flash'}, 'openai': {'model_name': 'gpt-4o'}}}


---
### ⚙️ Cell 3 — Load Configuration

This loads project config (API keys, model names, etc.) from a YAML/dotenv file via a custom `config_loader` utility.

> **Java parallel:** Similar to reading from `application.properties` / `application.yml` in Spring Boot using `@Value` or `@ConfigurationProperties`.

---


In [4]:
# ==========================================
# Load LLM
# ==========================================

from src.utils.model_loader import load_llm
llm = load_llm()

print("LLM Loaded Successfully\n")
print(type(llm))

LLM Loaded Successfully

<class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>


---
### 🧠 Cell 4 — Load the LLM

The LLM (Large Language Model) is the AI brain. `load_llm()` returns a model object (e.g. `ChatGoogleGenerativeAI` or `ChatOpenAI`).

> **Java parallel:** Think of this as getting a Spring `@Bean` from the ApplicationContext:
> ```java
> LLM llm = applicationContext.getBean(LLM.class);
> ```

The returned `llm` object has an `.invoke(prompt)` method — just like calling a method on a service object.

---


In [5]:
# ==========================================
# Verify LLM Connection
# ==========================================

response = llm.invoke("In one sentence, explain what LangChain is.")
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'LangChain is an open-source software framework that simplifies the process of building context-aware applications powered by large language models by integrating them with external data sources, APIs, and tools.', 'extras': {'signature': 'Er4OCrsOARFNMg8JSmCg3ON3ylK4ha7QSbS5hq8KTtHey5GHwt02Mu+6o49pmQ+X1U2o4xxiHdDmSNTYlQEAgeTw7eLqgaOoGsgfe6Tqe4TGo/85UtLFMc+iqLfSID3tP6hV2sNDDq3qGRcagBHEZYFJIsKfzkbN/plRdGhw2xxWbGKv6yWmNlZcr/oP/nn5NYO6FQcB0D3lk0JYZeN24QlCOem0nOGQPPRToGszCQbVXCdfUZe/hIuQDotquf3IbdL64aYkcGFfNsxBlJvJAFkAsu/zoN4wV+IAiT/z8Z2s4WipRWh9ar0tfOvjScA3GPHmW56P1F31EcPU6hRHqjGwGzE2/RQX1BWKM83/gYMvCIr847GvDkJeR3Kyf6hg1GzZ2hsHKw02hVsIigXypjcSpTG8IQIf8ZmGgn6w1DJ++ICXn0l7aWgrUpm//f0J+l4oxBuMCJpHrRVcWVFwpZQP1dCjWj+W13vkDuUPD8kwprr0kFJBlBS1tvGtHFSn7eecOBVo/NyAqSfVw2pEiCdymAfp/aTZhrM8VZIdtttIm+POq2ahesvM0bzzFRd5sTJF3D4eop5AyTyMLbJBhNKnE6GV5O2h6ohmlnhW4YOVd+ELODfPoFIj2h1bP+HBwSWT/4nhSbHuf5jfuqHMmFTTTNg+D2wQ8XsOQmjWcMKTL1JzuxHWeL1qOk4u0PofeAqEPXAru72EoZuee6VYtmMwDPTTHS

---
### 🔌 Cell 5 — Verify LLM Connection

A quick sanity-check: we send a one-sentence prompt to the LLM and print its response. 

> **Java parallel:** Like a `@PostConstruct` health-check or calling a REST endpoint with `RestTemplate` to verify the service is up.

```python
response = llm.invoke("In one sentence, explain what LangChain is.")
print(response.content)  # response is an AIMessage object with a .content field
```

---


In [6]:
# ==========================================
# Calculator Tool
# ==========================================

from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    Example:
    25 * 10
    100 / 4
    (10 + 5) * 2
    """

    result = eval(expression)

    return str(result)


print("Tool Name :", calculator.name)
print("Tool Description :\n")
print(calculator.description)

Tool Name : calculator
Tool Description :

Evaluate a mathematical expression.
Example:
25 * 10
100 / 4
(10 + 5) * 2


---
### 🔧 Cell 6 — Define a Custom Tool: `calculator`

This is where the magic starts! The `@tool` decorator converts a plain Python function into a **LangChain Tool** the agent can call.

> **Java parallel:** 
> ```java
> // Java: annotating a method as a tool
> @Tool(name = "calculator", description = "Evaluate a math expression")
> public String calculator(String expression) { ... }
> ```

#### How `@tool` works:
- The **function name** becomes the tool name
- The **docstring** becomes the tool description (the LLM reads this to decide when to use it!)
- The **type hints** (`expression: str`) become the tool's input schema

> ⚠️ **Note:** `eval()` executes the expression as Python code. This is fine for demos, but never use `eval()` on untrusted user input in production — it's a security risk (same as `Runtime.exec()` in Java).

---


In [7]:
# ==========================================
# Test Calculator Tool
# ==========================================

result = calculator.invoke("125 * 8")
print("Result:", result)

Result: 1000


---
### ✅ Cell 7 — Test the Calculator Tool Directly

Before wiring the tool into an agent, we test it standalone using `.invoke()`.

> **Java parallel:** Unit-testing your `@Service` method in isolation before integration testing.

```python
result = calculator.invoke("125 * 8")
# LangChain tools accept either a plain string or a dict of named args
```

---


## Run -> pip install ddgs

In [8]:
# ==========================================
# Create Search Tool
# ==========================================

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

print("Tool Created Successfully")
print(type(search_tool))

Tool Created Successfully
<class 'langchain_community.tools.ddg_search.tool.DuckDuckGoSearchRun'>


---
### 🌐 Cell 9 — Create a Web Search Tool (DuckDuckGo)

LangChain ships with many pre-built tools. `DuckDuckGoSearchRun` is a ready-made tool that searches the web.

> **Java parallel:** Using a pre-built Spring Integration adapter instead of writing your own HTTP client — like `spring-integration-http` for a REST API.

No `@tool` decorator needed here — `DuckDuckGoSearchRun` is already a LangChain tool class.

---


In [9]:
# ==========================================
# Test Search Tool
# ==========================================

result = search_tool.invoke("What is Agentic AI?")
print(result)

25 Nov 2025 · Comments · Stop Using AI Wrong — Agentic AI vs RAG Explained · Andrew Ng: The Biggest Opportunities in AI Aren't Where You Think · The Future of AI Agents with ... 2 May 2026 · Agentic AI is a type of AI system that can autonomously make decisions, plan actions and execute tasks to achieve specific goals with minimal human ... 18 Feb 2026 · AI agents or agentic AI, a new breed of AI systems that are semi- or fully autonomous and thus able to perceive, reason, and act on their own. 30 Jan 2026 · Agentic AI refers to intelligent systems composed of autonomous agents that can reason, act, and adapt to achieve goals with or without human oversight. Rating 9.3/10 (505) 22 Sept 2025 · Agentic AI means AI systems that autonomously plan and execute actions to achieve goals. Agentic AI monitors delivery systems, detects delays, and initiates ...


---
### 🧪 Cell 10 — Test the Search Tool

Same pattern as before: test the tool in isolation first.

---


In [10]:
# ==========================================
# Create Keyword Extractor Tool
# ==========================================

from langchain_core.tools import tool


@tool
def keyword_extractor(text: str) -> str:
    """
    Extract important keywords from a given text.
    """

    prompt = f"""
    Extract the 5 most important keywords from the text below.

    Text:
    {text}

    Return only the keywords as a comma separated list.
    """

    response = llm.invoke(prompt)

    return response.content


print("Tool Name :", keyword_extractor.name)
print("\nTool Description:\n")
print(keyword_extractor.description)

Tool Name : keyword_extractor

Tool Description:

Extract important keywords from a given text.


---
### 🔑 Cell 11 — Define an LLM-Powered Tool: `keyword_extractor`

This tool is interesting — it uses the **LLM itself** inside the tool! It crafts a prompt, calls `llm.invoke()`, and returns the result.

> **Java parallel:** An inner service that calls another service — like a `KeywordService` calling an `NlpService` internally.

#### Pattern:
```
Agent → calls keyword_extractor tool → tool calls LLM → returns keywords → Agent uses them
```

The `f-string` is Python's template string syntax:
```python
# Python f-string
f"Extract keywords from: {text}"

# Java equivalent (String.format or text blocks)
String.format("Extract keywords from: %s", text)
```

---


In [11]:
# ==========================================
# Test Keyword Extractor Tool
# ==========================================

result = keyword_extractor.invoke(
    """
    LangGraph is a framework for building stateful,
    multi-agent applications using large language models.
    It provides graph-based orchestration and workflow management.
    """
)

print(result)

[{'type': 'text', 'text': 'LangGraph, stateful, multi-agent, orchestration, workflow management', 'extras': {'signature': 'EpEXCo4XARFNMg9bdm7PZb8kz9ZNuuxBYIjQN+BHxfIxuZhvoswBbKXFch0AJu9FnjzHg+675ZvD67s9RAd9Be6gw+qJBPAeMxN8T5X48PZLsJEgNT/KSfZRka5irHo44BNU8oWGr0Haosu+/k/EeiHtps3DWWu2iUVV8GxMDm5kgRf8AmGTMC5yD/Y1ew1E0aS+k+z03rIAHrE/h2yiqQ0nf3d84pAMpWGPyTtvm5x7UwKyA9nERix6oGHacAF/NlIZMN9BtxqLMh1r7T7PKeTbFT+skLoDlCtWg4w84TqJngsWKYiGIzCnIAEs1lvXlzjy6GSpGfHYeEy6uVqmW6XKd4gOTgr3CdVj37+2YaOFigyxMRcpW6kGOo8BcMGXdPwq8RFjbpcS/hs2HiiRAYRkDTdkRzdK4cvr01jvx71qNy8JSQh30PT7AP/6m424NvdfmnyImixhInD5hNu/zUCdKi7qkk5m3jhhJRvLHSi1dgemhWCRYGz0SpIhUYvvgQX6MrWasj0HacxA3PSQLY5vGw22ck9BApwvJjuMql+h2DORKRaBH3hejAOQjxOF4YQ9izi0fFkJgiNYTOyaQ1qeu4sl9HY5UcR6gY37CN1B6Rxnmn3geGPuKSy4RhAtNrEWNixrYjL4bfLdgZxLq7xo9rEatmglpw8UV9Gk3dg3eaf6pukCHy+rOc9UlCGBEkB9Yly27EzEmgZvEkCLgoDiSHuiQ5mo/MaboZO49jotNZ3lia24snft5YaiC00UYY2e7E1gGaAZWaPeFFblNsePGpoviuEs0H84GeIXW8tDmUMp1OHAQ3DXh8cb8KMgGwBQlrYFx75DyUoT+I4vILlCnAqpHTrvygPYwyyA+ufiC

---
### 🧪 Cell 12 — Test the Keyword Extractor

Testing the LLM-powered tool standalone.

> Notice `.invoke()` accepts a plain string here — LangChain maps it to the first (and only) parameter automatically.

---


In [12]:
# ==========================================
# Create Research Summary Tool
# ==========================================

from langchain_core.tools import tool


@tool
def research_summary_tool(topic: str) -> str:
    """
    Generate a concise research summary for a given topic.
    """

    prompt = f"""
    You are an AI research assistant.

    Generate a concise research summary on:

    Topic: {topic}

    Requirements:
    - Maximum 150 words
    - Professional tone
    - Mention key concepts
    - Mention practical applications
    """

    response = llm.invoke(prompt)

    return response.content


print("Tool Name :", research_summary_tool.name)
print("\nTool Description:\n")
print(research_summary_tool.description)

Tool Name : research_summary_tool

Tool Description:

Generate a concise research summary for a given topic.


---
### 📝 Cell 13 — Define a Research Summary Tool

Another LLM-powered tool, but with a richer **system prompt** defining the AI's persona and output requirements.

> This is the **Prompt Engineering** pattern: carefully crafting instructions to get consistent, high-quality output from the LLM — like specifying a precise API contract.

#### Key prompt techniques used:
- Role assignment: `"You are an AI research assistant"`
- Constraints: `"Maximum 150 words"`
- Structure: bullet-point requirements

---


In [13]:
# ==========================================
# Test Research Summary Tool
# ==========================================

result = research_summary_tool.invoke("LangGraph")
print(result)

[{'type': 'text', 'text': '**LangGraph** is an open-source framework developed by LangChain for orchestrating complex, stateful multi-agent LLM applications. Unlike linear chains or strict Directed Acyclic Graphs (DAGs), LangGraph natively supports cyclic computational flows, enabling agents to perform iterative loops, reflection, and self-correction.\n\nKey concepts center on a graph-based architecture comprising **Nodes** (agent actions or tools), **Edges** (conditional control-flow logic), and a synchronized **State** object. Its built-in persistence layer enables enterprise-grade features such as human-in-the-loop approval gates, fault tolerance, and execution "time-travel" (rewinding state).\n\nPractically, LangGraph is designed for building sophisticated autonomous systems. Common applications include self-correcting code generation pipelines, collaborative multi-agent research tools, dynamic customer support bots, and long-running business process automations requiring precise c

---
### 🧪 Cell 14 — Test the Research Summary Tool

---


In [14]:
# ==========================================
# Register All Tools
# ==========================================

tools = [
    calculator,
    search_tool,
    keyword_extractor,
    research_summary_tool
]

print(f"Total Tools Registered: {len(tools)}")

for tool_obj in tools:
    print("-", tool_obj.name)

Total Tools Registered: 4
- calculator
- duckduckgo_search
- keyword_extractor
- research_summary_tool


---
### 📋 Cell 15 — Register All Tools

We collect all tools into a Python `list` — equivalent to a Java `List<Tool>`.

> **Java parallel:** Registering beans in a Spring ApplicationContext or building a `Map<String, Handler>` for a command-pattern dispatcher.

The agent will inspect this list to know what capabilities it has.

---


In [15]:
# ==========================================
# Create Agent Prompt
# ==========================================

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an AI Research Assistant.

            You have access to several tools.

            Use tools whenever necessary.

            For:
            - Calculations -> use calculator
            - Web search -> use search tool
            - Keyword extraction -> use keyword_extractor
            - Research summaries -> use research_summary_tool

            Always provide clear and professional answers.
            """
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}")
    ]
)

print("Prompt Created Successfully")

Prompt Created Successfully


---
### 💬 Cell 16 — Create the Agent Prompt

A `ChatPromptTemplate` defines the **conversation structure** sent to the LLM:

| Slot | Purpose |
|------|---------|
| `system` | Sets the AI's persona and rules (like a constructor / class-level config) |
| `human` | The user's input (`{input}` is a placeholder filled at runtime) |
| `placeholder` | `{agent_scratchpad}` — internal workspace for the agent's reasoning steps |

> **Java parallel:** This is like a `MessageTemplate` or `EmailTemplate` with `${variable}` placeholders filled in at runtime.

---


In [16]:
# ==========================================
# Create Tool Calling Agent
# ==========================================

from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

print("Tool Calling Agent Created Successfully")

ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (d:\projects\python\stepIntoAI\LangChain_05-tool-calling\lngcntc\Lib\site-packages\langchain\agents\__init__.py)

---
### 🤖 Cell 17 — Create the Tool-Calling Agent

This is the **orchestrator** — it combines the LLM, tools, and prompt into a working agent.

```python
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
```

#### Two objects, two responsibilities:

| Object | Role | Java parallel |
|--------|------|---------------|
| `agent` | Decision-maker: what tool to call next | `DispatcherServlet` / Strategy pattern |
| `agent_executor` | Execution loop: actually calls the tools | `ExecutorService` / `CommandBus` |

> `verbose=True` prints the agent's reasoning chain — great for learning! It shows you **which tool** the LLM decided to call and **why**.

#### How it works internally:
```
User Input
    ↓
LLM decides: "I need to call calculator with '125 * 75'"
    ↓
AgentExecutor calls calculator.invoke("125 * 75") → "9375"
    ↓
LLM sees result, generates final answer
    ↓
Output
```

---


In [17]:
# ==========================================
# Test Simple Tool Calling
# ==========================================

response = agent_executor.invoke({"input": "What is 125 * 75?"})

print("\nFinal Answer:\n")
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '125 * 75'}`


9375The result of \(125 \times 75\) is 9,375.

> Finished chain.

Final Answer:

The result of \(125 \times 75\) is 9,375.


---
### 🚀 Cell 18 — Test 1: Simple Calculation

The agent receives: `"What is 125 * 75?"`

With `verbose=True`, watch the output — you'll see the LLM:
1. Decide to use the `calculator` tool
2. Pass `"125 * 75"` as the expression
3. Get back `"9375"`
4. Format a final human-readable answer

> **Key insight:** The LLM didn't calculate `125 * 75` itself — it *delegated* to a tool, just like a good engineer who doesn't reinvent the wheel.

---


In [18]:
response = agent_executor.invoke(
    {
        "input": """
        Search the web for the latest trends in Agentic AI
        and provide a concise summary.
        """
    }
)

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': 'latest trends in Agentic AI 2023'}`




c:\Users\Personal\Downloads\Learnyard-MLOps\GenAI-For-Dev-Course\18-ToolCalling-LangChain\myenv2\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
c:\Users\Personal\Downloads\Learnyard-MLOps\GenAI-For-Dev-Course\18-ToolCalling-LangChain\myenv2\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:64: UserWarning: backend='api' is deprecated, using backend='auto'
  ddgs_gen = ddgs.text(


No good DuckDuckGo Search Result was found
Invoking: `research_summary` with `{'topic': 'Agentic AI trends'}`


Agentic AI refers to artificial intelligence systems designed with autonomous decision-making capabilities, enabling them to perform tasks without human intervention. Recent trends in agentic AI focus on enhancing decision-making processes through advanced machine learning algorithms, reinforcement learning, and neural networks. Key concepts include autonomy, adaptability, and ethical considerations, with an emphasis on creating systems that can learn contextually and evolve based on environmental feedback. Practical applications are vast, encompassing autonomous vehicles, personalized virtual assistants, smart manufacturing systems, and adaptive cybersecurity solutions. The integration of agentic AI in these areas aims to improve efficiency, reduce human error, and enable more sophisticated interactions between humans and machines. However, the rise of agentic AI also necess

---
### 🌐 Cell 19 — Test 2: Web Search + Summary

Multi-step task: the agent will:
1. Use `DuckDuckGoSearchRun` to fetch live web results
2. Summarize what it found

> This shows the agent **chaining** tools across multiple steps — similar to a Java pipeline or `CompletableFuture.thenApply()` chain.

---


In [19]:
response = agent_executor.invoke(
    {
        "input": """
        Extract important keywords from the following text
        and then provide a concise research summary.

        LangGraph is a framework for building stateful
        multi-agent applications using LLMs.
        """
    }
)

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `keyword_extractor` with `{'text': 'LangGraph is a framework for building stateful multi-agent applications using LLMs.'}`


LangGraph, framework, stateful, multi-agent, LLMs
Invoking: `research_summary` with `{'topic': 'LangGraph framework for building stateful multi-agent applications using LLMs'}`


The LangGraph framework is an innovative approach for developing stateful multi-agent applications using Large Language Models (LLMs). Central to LangGraph is its ability to facilitate complex interactions among multiple AI agents by maintaining a stateful environment, which enhances communication coherence and task coordination. Key concepts include state management, which ensures continuity and context retention across interactions, and agent orchestration, enabling dynamic role assignments and task prioritization. LangGraph leverages the capabilities of LLMs to process and generate human-like responses, allowing agents to effectively 

---
### 🔗 Cell 20 — Test 3: Multi-Tool Chain (Keyword Extract → Research Summary)

The most advanced test: the agent needs to:
1. Call `keyword_extractor` on the input text
2. Use those keywords to call `research_summary_tool`

> **Java parallel:** A multi-step service chain — `KeywordService.extract()` → `ResearchService.summarize()` — orchestrated automatically by the agent, not by hardcoded `if/else` logic.

This is the **power of LLM-driven orchestration**: you describe *what* you want, and the agent figures out *how* to achieve it using available tools.

---
